In [1]:
"""
INDIA.RUNS AI Hiring Challenge

Objective:
Develop an AI-powered candidate ranking system that recommends
the most suitable applicants for a given job description.

Approach:
1. Extract and preprocess candidate profiles.
2. Perform rule-based scoring using skills, experience,
   job title, and company information.
3. Compute semantic similarity using TF-IDF and Cosine Similarity.
4. Combine both approaches into a hybrid ranking model.
5. Generate explainable recommendations and export the top candidates
   in the required submission format.
"""

'\nINDIA.RUNS AI Hiring Challenge\n\nObjective:\nDevelop an AI-powered candidate ranking system that recommends\nthe most suitable applicants for a given job description.\n\nApproach:\n1. Extract and preprocess candidate profiles.\n2. Perform rule-based scoring using skills, experience,\n   job title, and company information.\n3. Compute semantic similarity using TF-IDF and Cosine Similarity.\n4. Combine both approaches into a hybrid ranking model.\n5. Generate explainable recommendations and export the top candidates\n   in the required submission format.\n'

In [2]:
# Load the job description from the Word document.
# This serves as the reference profile against which all candidates
# will be evaluated for relevance.
from docx import Document

doc = Document("job_description.docx")

jd_text = ""

for para in doc.paragraphs:
    jd_text += para.text + " "

print(jd_text[:1000])

Job Description: Senior AI Engineer — Founding Team Company: Redrob AI (Series A AI-native talent intelligence platform) Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities Employment Type: Full-time Experience Required: 5–9 years (see "what we mean by this" below)  Let's be honest about this role We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we're going to tell you what we actually need and what we've gotten wrong before. If you've spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn't it. If you've spent your career bouncing between early-stage startups and you want to "just code" without having to think 

In [3]:
# Read the candidate dataset in JSONL format.
# Inspect one sample record to understand the available fields
# before preprocessing.
import json

with open("candidates.jsonl", "r", encoding="utf-8") as f:
    first_candidate = json.loads(next(f))

first_candidate

{'candidate_id': 'CAND_0000001',
 'profile': {'anonymized_name': 'Ira Vora',
  'headline': 'Backend Engineer | SQL, Spark, Cloud',
  'summary': "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.",
  'location': 'Toronto',
  'country': 'Canada',
  'years_of_experience': 6.9,
  'current_title': 'Backend Engineer',
  'current_company': 'Mindtree',
  'current_company_size': '10001+',
  'current_industry': 'IT Servic

In [4]:
print(first_candidate.keys())


dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])


In [5]:
# Convert the job description into CSV format for easier processing
# and compatibility with downstream data pipelines.
import pandas as pd

doc = Document("job_description.docx")

jd_text = "\n".join([para.text for para in doc.paragraphs])

jd_df = pd.DataFrame({
    "job_description": [jd_text]
})

jd_df.to_csv("job_description.csv",index=False)
jd_df.head()

,job_description
0,Job Description: Senior AI Engineer — Founding...


In [6]:
first_candidate["profile"]

{'anonymized_name': 'Ira Vora',
 'headline': 'Backend Engineer | SQL, Spark, Cloud',
 'summary': "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.",
 'location': 'Toronto',
 'country': 'Canada',
 'years_of_experience': 6.9,
 'current_title': 'Backend Engineer',
 'current_company': 'Mindtree',
 'current_company_size': '10001+',
 'current_industry': 'IT Services'}

In [7]:
# Extract important candidate information including profile summary,
# skills, work experience, current designation, and company.
# Store everything in a structured DataFrame for analysis.
import json

records = []

with open("candidates.jsonl", "r", encoding="utf-8") as f:

    for line in f:

        candidate = json.loads(line)

        profile = candidate.get("profile", {})

        skills = ", ".join(
            skill.get("name", "")
            for skill in candidate.get("skills", [])
        )

        records.append({
            "candidate_id": candidate.get("candidate_id", ""),
            "headline": profile.get("headline", ""),
            "summary": profile.get("summary", ""),
            "years_of_experience": profile.get("years_of_experience", ""),
            "current_title": profile.get("current_title", ""),
            "current_company": profile.get("current_company", ""),
            "skills": skills
        })

df = pd.DataFrame(records)

print(df.shape)

df.head()

(100000, 7)


,candidate_id,headline,summary,years_of_experience,current_title,current_company,skills
0,CAND_0000001,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,6.9,Backend Engineer,Mindtree,"Tailwind, NLP, Image Classification, Fine-tuni..."
1,CAND_0000002,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,12.5,Operations Manager,Wipro,"Project Management, React, Photoshop, TypeScri..."
2,CAND_0000003,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,1.1,Customer Support,TCS,"Angular, SEO, Excel, Accounting, Kubernetes, D..."
3,CAND_0000004,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,3.8,Marketing Manager,Dunder Mifflin,"Node.js, Content Writing, Redux, Airflow, Grap..."
4,CAND_0000005,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,11.0,Accountant,Stark Industries,"SQL, PowerPoint, Photoshop, Tailwind, Apache F..."


In [8]:
df.shape

(100000, 7)

In [9]:
# Combine relevant profile fields into a single searchable text column.
# This consolidated representation improves keyword matching
# and semantic similarity calculations.
df["tags"] = (
    df["headline"].fillna("")+ " " +
    df["summary"].fillna("") + " " +
    df["skills"].fillna("") + " " +
    df["current_title"].fillna("") + " " +
    df["current_company"].fillna("")
).str.lower()
df.head()

,candidate_id,headline,summary,years_of_experience,current_title,current_company,skills,tags
0,CAND_0000001,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,6.9,Backend Engineer,Mindtree,"Tailwind, NLP, Image Classification, Fine-tuni...","backend engineer | sql, spark, cloud software ..."
1,CAND_0000002,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,12.5,Operations Manager,Wipro,"Project Management, React, Photoshop, TypeScri...",operations manager | 12.5+ yrs experience prof...
2,CAND_0000003,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,1.1,Customer Support,TCS,"Angular, SEO, Excel, Accounting, Kubernetes, D...",customer support | 1.1+ yrs experience profess...
3,CAND_0000004,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,3.8,Marketing Manager,Dunder Mifflin,"Node.js, Content Writing, Redux, Airflow, Grap...",marketing manager | driving business outcomes ...
4,CAND_0000005,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,11.0,Accountant,Stark Industries,"SQL, PowerPoint, Photoshop, Tailwind, Apache F...",accountant | helping teams scale professional ...


In [10]:
jd_df

,job_description
0,Job Description: Senior AI Engineer — Founding...


In [11]:
jd_text

'Job Description: Senior AI Engineer — Founding Team\nCompany: Redrob AI (Series A AI-native talent intelligence platform)\nLocation: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities\nEmployment Type: Full-time\nExperience Required: 5–9 years (see "what we mean by this" below)\n\nLet\'s be honest about this role\nWe\'re going to write this JD differently from most. We\'re a Series A company that just raised our round and we\'re building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we\'re going to tell you what we actually need and what we\'ve gotten wrong before.\nIf you\'ve spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn\'t it.\nIf you\'ve spent your career bouncing between early-stage startups and you want to "just code" witho

In [12]:
# Define the key technical skills expected for the role
# based on the job description.
# These skills will be used for rule-based candidate scoring.
target_skills = [
    "python",
    "llm",
    "nlp",
    "embedding",
    "retrieval",
    "vector",
    "faiss",
    "pinecone",
    "qdrant",
    "milvus",
    "ranking",
    "recommendation",
    "machine learning",
    "deep learning",
    "rag"
]

In [13]:
# Calculate a skill score by counting how many required skills
# are present in each candidate's profile.
def skill_score(text):

    text = str(text).lower()

    score = 0

    for skill in target_skills:

        if skill in text:
            score += 1

    return score

In [14]:
# Assign scores based on years of professional experience.
# Candidates within the preferred experience range receive
# higher scores.
def experience_score(years):

    try:
        years = float(years)

        if 5 <= years <= 9:
            return 10

        elif 4 <= years <= 12:
            return 7

        else:
            return 3

    except:
        return 0

In [15]:
# Reward candidates whose current designation closely matches
# the target job role such as AI Engineer, ML Engineer,
# NLP Engineer, or Data Scientist.
target_titles = [
    "ai engineer",
    "ml engineer",
    "machine learning engineer",
    "data scientist",
    "nlp engineer"
]

def title_score(title):

    title = str(title).lower()

    for t in target_titles:
        if t in title:
            return 10

    return 0

In [16]:
# Provide a small bonus score to candidates currently working
# at well-known technology companies.
target_companies = [ "google","meta","microsoft","amazon","apple","netflix","openai","anthropic","hugging face","nvidia"]


def company_score(company):

    company = str(company).lower()

    for c in target_companies:
        if c in company:
            return 10

    return 0


In [17]:
# Calculate a weighted rule-based score using
# skills, experience, job title, and company information.
df["skill_score"] = df["tags"].apply(skill_score)


df["exp_score"] = df["years_of_experience"].apply(experience_score)

df["title_score"] = df["current_title"].apply(title_score)
df["company_score"] = df["current_company"].apply(company_score)

df["final_score"] = (
    0.4 * df["skill_score"] +
    0.38 * df["exp_score"] +
    0.2 * df["title_score"] +
    0.02 * df["company_score"]
    )

In [18]:
df.columns

Index(['candidate_id', 'headline', 'summary', 'years_of_experience',
       'current_title', 'current_company', 'skills', 'tags', 'skill_score',
       'exp_score', 'title_score', 'company_score', 'final_score'],
      dtype='object')

In [19]:
# This provides the initial ranking of candidates.
df[['candidate_id',
    'current_title',
    'years_of_experience',
    'skill_score',
    'exp_score',
    'title_score',
    'company_score',
    'final_score'
]].sort_values(
    by='final_score',
    ascending=False
).head(20)

,candidate_id,current_title,years_of_experience,skill_score,exp_score,title_score,company_score,final_score
60071,CAND_0060072,Staff Machine Learning Engineer,5.7,12,10,10,10,10.8
62246,CAND_0062247,AI Engineer,7.3,11,10,10,10,10.4
2024,CAND_0002025,Senior AI Engineer,5.9,11,10,10,10,10.4
18498,CAND_0018499,Senior Machine Learning Engineer,7.2,11,10,10,0,10.2
81845,CAND_0081846,Lead AI Engineer,6.7,11,10,10,0,10.2
16800,CAND_0016801,ML Engineer,5.3,11,10,10,0,10.2
70397,CAND_0070398,Machine Learning Engineer,7.2,11,10,10,0,10.2
77336,CAND_0077337,Staff Machine Learning Engineer,7.0,11,10,10,0,10.2
13392,CAND_0013393,ML Engineer,5.3,11,10,10,0,10.2
45140,CAND_0045141,Data Scientist,5.3,11,10,10,0,10.2


In [20]:
top_candidates = df.sort_values(
    by='final_score',
    ascending=False
)

top_candidates[
    ['candidate_id',
     'current_title',
     'current_company',
     'years_of_experience',
     'skills',
     'final_score']
].head(10)

,candidate_id,current_title,current_company,years_of_experience,skills,final_score
60071,CAND_0060072,Staff Machine Learning Engineer,Amazon,5.7,"Deep Learning, Sentence Transformers, PyTorch,...",10.8
62246,CAND_0062247,AI Engineer,Google,7.3,"Image Classification, OpenCV, Reinforcement Le...",10.4
2024,CAND_0002025,Senior AI Engineer,Apple,5.9,"Diffusion Models, FAISS, TensorFlow, scikit-le...",10.4
18498,CAND_0018499,Senior Machine Learning Engineer,Zomato,7.2,"Deep Learning, Weaviate, Recommendation System...",10.2
81845,CAND_0081846,Lead AI Engineer,Razorpay,6.7,"Data Science, Information Retrieval, LlamaInde...",10.2
16800,CAND_0016801,ML Engineer,Infosys,5.3,"scikit-learn, TypeScript, pgvector, Image Clas...",10.2
70397,CAND_0070398,Machine Learning Engineer,Genpact AI,7.2,"Image Classification, Data Science, BM25, RAG,...",10.2
77336,CAND_0077337,Staff Machine Learning Engineer,Paytm,7.0,"GANs, Semantic Search, QLoRA, pgvector, Pineco...",10.2
13392,CAND_0013393,ML Engineer,Freshworks,5.3,"Excel, Embeddings, MLflow, Pinecone, Qdrant, D...",10.2
45140,CAND_0045141,Data Scientist,HCL,5.3,"YOLO, Microservices, TTS, Recommendation Syste...",10.2


In [21]:
# Convert both the job description and candidate profiles
# into TF-IDF vectors to capture textual importance
# across documents.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
documents = [jd_text] + df["tags"].tolist()


In [23]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

In [24]:
tfidf_matrix = vectorizer.fit_transform(documents)
tfidf_matrix


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5696671 stored elements and shape (100001, 952)>

In [25]:
# Compute cosine similarity between the job description
# and every candidate profile to measure semantic relevance.
jd_vector = tfidf_matrix[0]

candidate_vectors = tfidf_matrix[1:]

similarity_scores = cosine_similarity(
    jd_vector,
    candidate_vectors
).flatten()

df["similarity_score"] = similarity_scores


In [26]:
# Combine the rule-based score with semantic similarity
# to generate a more balanced and accurate candidate ranking.
df["hybrid_score"] = (
    0.7 * df["final_score"]
    +
    0.3 * df["similarity_score"] * 10
)

In [27]:
# Sort candidates according to the hybrid score
# and retain the highest-ranked candidates.
top_candidates = df.sort_values(
    by="hybrid_score",
    ascending=False
)

top_candidates[
    [
        "candidate_id",
        "current_title",
        "years_of_experience",
        "hybrid_score"
    ]
].head(20)


,candidate_id,current_title,years_of_experience,hybrid_score
60071,CAND_0060072,Staff Machine Learning Engineer,5.7,8.200490
2024,CAND_0002025,Senior AI Engineer,5.9,8.037388
77336,CAND_0077337,Staff Machine Learning Engineer,7.0,7.816436
33860,CAND_0033861,Senior NLP Engineer,8.0,7.812209
81845,CAND_0081846,Lead AI Engineer,6.7,7.804282
62246,CAND_0062247,AI Engineer,7.3,7.787675
18498,CAND_0018499,Senior Machine Learning Engineer,7.2,7.753812
88024,CAND_0088025,Staff Machine Learning Engineer,8.6,7.740267
46063,CAND_0046064,Senior NLP Engineer,8.9,7.534054
70397,CAND_0070398,Machine Learning Engineer,7.2,7.526423


In [28]:
sample = pd.read_csv("sample_submission.csv")

sample.head()

print(sample.columns)

print(sample.shape)

Index(['candidate_id', 'rank', 'score', 'reasoning'], dtype='object')
(100, 4)


In [29]:
sample.head()

,candidate_id,rank,score,reasoning
0,CAND_0004989,1,0.992,HR Manager with 6.1 yrs; 9 AI core skills; res...
1,CAND_0001195,2,0.984,HR Manager with 8.7 yrs; 9 AI core skills; res...
2,CAND_0003114,3,0.976,ML Engineer with 6.4 yrs; 4 AI core skills; re...
3,CAND_0000339,4,0.968,Content Writer with 8.3 yrs; 8 AI core skills;...
4,CAND_0001082,5,0.960,HR Manager with 5.0 yrs; 8 AI core skills; res...


In [30]:
top100 = df.sort_values(
    by="hybrid_score",
    ascending=False
).head(100).copy()

top100.reset_index(drop=True, inplace=True)
top100.index = top100.index + 1
top100

,candidate_id,headline,summary,years_of_experience,current_title,current_company,skills,tags,skill_score,exp_score,title_score,company_score,final_score,similarity_score,hybrid_score
1,CAND_0060072,"Staff Machine Learning Engineer | LLMs, RAG, V...",Senior AI engineer with 5.7 years of hands-on ...,5.7,Staff Machine Learning Engineer,Amazon,"Deep Learning, Sentence Transformers, PyTorch,...","staff machine learning engineer | llms, rag, v...",12,10,10,10,10.8,0.213497,8.200490
2,CAND_0002025,Senior AI Engineer | Building AI-native search...,Senior AI engineer with 5.9 years of hands-on ...,5.9,Senior AI Engineer,Apple,"Diffusion Models, FAISS, TensorFlow, scikit-le...",senior ai engineer | building ai-native search...,11,10,10,10,10.4,0.252463,8.037388
3,CAND_0077337,Staff Machine Learning Engineer | Building AI-...,Senior AI engineer with 7.0 years of hands-on ...,7.0,Staff Machine Learning Engineer,Paytm,"GANs, Semantic Search, QLoRA, pgvector, Pineco...",staff machine learning engineer | building ai-...,11,10,10,0,10.2,0.225479,7.816436
4,CAND_0033861,Senior NLP Engineer | Building AI-native searc...,Senior AI engineer with 8.0 years of hands-on ...,8.0,Senior NLP Engineer,Mad Street Den,"Reinforcement Learning, Weaviate, LoRA, LLMs, ...",senior nlp engineer | building ai-native searc...,11,10,10,0,10.2,0.224070,7.812209
5,CAND_0081846,"Lead AI Engineer | LLMs, RAG, Vector Search | ...",Senior AI engineer with 6.7 years of hands-on ...,6.7,Lead AI Engineer,Razorpay,"Data Science, Information Retrieval, LlamaInde...","lead ai engineer | llms, rag, vector search | ...",11,10,10,0,10.2,0.221427,7.804282
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,CAND_0095567,Data Scientist | 5.1 yrs in analytics & ML,Data scientist / ML engineer with 5.1 years of...,5.1,Data Scientist,Genpact AI,"PyTorch, Milvus, Weaviate, REST APIs, Diffusio...",data scientist | 5.1 yrs in analytics & ml dat...,8,10,10,0,9.0,0.093695,6.581084
97,CAND_0006567,Senior Engineer | Information Retrieval at scale,Senior engineer who has spent the last several...,7.9,Senior AI Engineer,Meta,"Speech Recognition, scikit-learn, Search Backe...",senior engineer | information retrieval at sca...,6,10,10,10,8.4,0.230923,6.572770
98,CAND_0019159,ML Engineer | Data Science & ML enthusiast,Data scientist / ML engineer with 6.1 years of...,6.1,ML Engineer,Genpact AI,"Spark, Vector Search, Forecasting, Machine Lea...",ml engineer | data science & ml enthusiast dat...,8,10,10,0,9.0,0.090002,6.570006
99,CAND_0073917,Data Scientist | 5.8 yrs in analytics & ML,Data scientist / ML engineer with 5.8 years of...,5.8,Data Scientist,InMobi,"BentoML, dbt, Rust, Learning to Rank, Computer...",data scientist | 5.8 yrs in analytics & ml dat...,8,10,10,0,9.0,0.089773,6.569320


In [31]:
top100["rank"] = range(1, 101)
top100

,candidate_id,headline,summary,years_of_experience,current_title,current_company,skills,tags,skill_score,exp_score,title_score,company_score,final_score,similarity_score,hybrid_score,rank
1,CAND_0060072,"Staff Machine Learning Engineer | LLMs, RAG, V...",Senior AI engineer with 5.7 years of hands-on ...,5.7,Staff Machine Learning Engineer,Amazon,"Deep Learning, Sentence Transformers, PyTorch,...","staff machine learning engineer | llms, rag, v...",12,10,10,10,10.8,0.213497,8.200490,1
2,CAND_0002025,Senior AI Engineer | Building AI-native search...,Senior AI engineer with 5.9 years of hands-on ...,5.9,Senior AI Engineer,Apple,"Diffusion Models, FAISS, TensorFlow, scikit-le...",senior ai engineer | building ai-native search...,11,10,10,10,10.4,0.252463,8.037388,2
3,CAND_0077337,Staff Machine Learning Engineer | Building AI-...,Senior AI engineer with 7.0 years of hands-on ...,7.0,Staff Machine Learning Engineer,Paytm,"GANs, Semantic Search, QLoRA, pgvector, Pineco...",staff machine learning engineer | building ai-...,11,10,10,0,10.2,0.225479,7.816436,3
4,CAND_0033861,Senior NLP Engineer | Building AI-native searc...,Senior AI engineer with 8.0 years of hands-on ...,8.0,Senior NLP Engineer,Mad Street Den,"Reinforcement Learning, Weaviate, LoRA, LLMs, ...",senior nlp engineer | building ai-native searc...,11,10,10,0,10.2,0.224070,7.812209,4
5,CAND_0081846,"Lead AI Engineer | LLMs, RAG, Vector Search | ...",Senior AI engineer with 6.7 years of hands-on ...,6.7,Lead AI Engineer,Razorpay,"Data Science, Information Retrieval, LlamaInde...","lead ai engineer | llms, rag, vector search | ...",11,10,10,0,10.2,0.221427,7.804282,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,CAND_0095567,Data Scientist | 5.1 yrs in analytics & ML,Data scientist / ML engineer with 5.1 years of...,5.1,Data Scientist,Genpact AI,"PyTorch, Milvus, Weaviate, REST APIs, Diffusio...",data scientist | 5.1 yrs in analytics & ml dat...,8,10,10,0,9.0,0.093695,6.581084,96
97,CAND_0006567,Senior Engineer | Information Retrieval at scale,Senior engineer who has spent the last several...,7.9,Senior AI Engineer,Meta,"Speech Recognition, scikit-learn, Search Backe...",senior engineer | information retrieval at sca...,6,10,10,10,8.4,0.230923,6.572770,97
98,CAND_0019159,ML Engineer | Data Science & ML enthusiast,Data scientist / ML engineer with 6.1 years of...,6.1,ML Engineer,Genpact AI,"Spark, Vector Search, Forecasting, Machine Lea...",ml engineer | data science & ml enthusiast dat...,8,10,10,0,9.0,0.090002,6.570006,98
99,CAND_0073917,Data Scientist | 5.8 yrs in analytics & ML,Data scientist / ML engineer with 5.8 years of...,5.8,Data Scientist,InMobi,"BentoML, dbt, Rust, Learning to Rank, Computer...",data scientist | 5.8 yrs in analytics & ml dat...,8,10,10,0,9.0,0.089773,6.569320,99


In [32]:
# Normalize hybrid scores into the required range
# using Min-Max Scaling for submission.
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0.8, 1.0))

top100["score"] = scaler.fit_transform(
    top100[["hybrid_score"]]
)


In [33]:
# Automatically generate a brief explanation
# describing why each candidate is recommended,
# highlighting experience, skills, and role alignment.
def generate_reasoning(row):

    title = row["current_title"]
    company=row["current_company"]
    yrs = round(row["years_of_experience"],1)

    skills = str(row["skills"]).split(",")

    top_skills = ", ".join(skills[:5])

    return (
        f"{title} with {yrs} years experience. "
        f"Key skills: {', '.join(skills)}. "
        f"Strong alignment with AI engineering, retrieval, ranking, and LLM-focused requirements."
    )

In [34]:
top100["reasoning"] = top100.apply(
    generate_reasoning,
    axis=1
)

In [35]:
top100

,candidate_id,headline,summary,years_of_experience,current_title,current_company,skills,tags,skill_score,exp_score,title_score,company_score,final_score,similarity_score,hybrid_score,rank,score,reasoning
1,CAND_0060072,"Staff Machine Learning Engineer | LLMs, RAG, V...",Senior AI engineer with 5.7 years of hands-on ...,5.7,Staff Machine Learning Engineer,Amazon,"Deep Learning, Sentence Transformers, PyTorch,...","staff machine learning engineer | llms, rag, v...",12,10,10,10,10.8,0.213497,8.200490,1,1.000000,Staff Machine Learning Engineer with 5.7 years...
2,CAND_0002025,Senior AI Engineer | Building AI-native search...,Senior AI engineer with 5.9 years of hands-on ...,5.9,Senior AI Engineer,Apple,"Diffusion Models, FAISS, TensorFlow, scikit-le...",senior ai engineer | building ai-native search...,11,10,10,10,10.4,0.252463,8.037388,2,0.980055,Senior AI Engineer with 5.9 years experience. ...
3,CAND_0077337,Staff Machine Learning Engineer | Building AI-...,Senior AI engineer with 7.0 years of hands-on ...,7.0,Staff Machine Learning Engineer,Paytm,"GANs, Semantic Search, QLoRA, pgvector, Pineco...",staff machine learning engineer | building ai-...,11,10,10,0,10.2,0.225479,7.816436,3,0.953035,Staff Machine Learning Engineer with 7.0 years...
4,CAND_0033861,Senior NLP Engineer | Building AI-native searc...,Senior AI engineer with 8.0 years of hands-on ...,8.0,Senior NLP Engineer,Mad Street Den,"Reinforcement Learning, Weaviate, LoRA, LLMs, ...",senior nlp engineer | building ai-native searc...,11,10,10,0,10.2,0.224070,7.812209,4,0.952518,Senior NLP Engineer with 8.0 years experience....
5,CAND_0081846,"Lead AI Engineer | LLMs, RAG, Vector Search | ...",Senior AI engineer with 6.7 years of hands-on ...,6.7,Lead AI Engineer,Razorpay,"Data Science, Information Retrieval, LlamaInde...","lead ai engineer | llms, rag, vector search | ...",11,10,10,0,10.2,0.221427,7.804282,5,0.951549,Lead AI Engineer with 6.7 years experience. Ke...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,CAND_0095567,Data Scientist | 5.1 yrs in analytics & ML,Data scientist / ML engineer with 5.1 years of...,5.1,Data Scientist,Genpact AI,"PyTorch, Milvus, Weaviate, REST APIs, Diffusio...",data scientist | 5.1 yrs in analytics & ml dat...,8,10,10,0,9.0,0.093695,6.581084,96,0.801967,Data Scientist with 5.1 years experience. Key ...
97,CAND_0006567,Senior Engineer | Information Retrieval at scale,Senior engineer who has spent the last several...,7.9,Senior AI Engineer,Meta,"Speech Recognition, scikit-learn, Search Backe...",senior engineer | information retrieval at sca...,6,10,10,10,8.4,0.230923,6.572770,97,0.800951,Senior AI Engineer with 7.9 years experience. ...
98,CAND_0019159,ML Engineer | Data Science & ML enthusiast,Data scientist / ML engineer with 6.1 years of...,6.1,ML Engineer,Genpact AI,"Spark, Vector Search, Forecasting, Machine Lea...",ml engineer | data science & ml enthusiast dat...,8,10,10,0,9.0,0.090002,6.570006,98,0.800613,ML Engineer with 6.1 years experience. Key ski...
99,CAND_0073917,Data Scientist | 5.8 yrs in analytics & ML,Data scientist / ML engineer with 5.8 years of...,5.8,Data Scientist,InMobi,"BentoML, dbt, Rust, Learning to Rank, Computer...",data scientist | 5.8 yrs in analytics & ml dat...,8,10,10,0,9.0,0.089773,6.569320,99,0.800529,Data Scientist with 5.8 years experience. Key ...


In [36]:
# Create the final submission containing:
# Candidate ID
# Rank
# Normalized Score
# AI-generated Reasoning
submission = top100[
    [
        "candidate_id",
        "rank",
        "score",
        "reasoning"
    ]
]

submission.head()

,candidate_id,rank,score,reasoning
1,CAND_0060072,1,1.000000,Staff Machine Learning Engineer with 5.7 years...
2,CAND_0002025,2,0.980055,Senior AI Engineer with 5.9 years experience. ...
3,CAND_0077337,3,0.953035,Staff Machine Learning Engineer with 7.0 years...
4,CAND_0033861,4,0.952518,Senior NLP Engineer with 8.0 years experience....
5,CAND_0081846,5,0.951549,Lead AI Engineer with 6.7 years experience. Ke...


In [37]:
# Export the final ranked candidate list
# as submission.csv for hackathon evaluation.
submission.to_csv(
    "submission.csv",
    index=False
)
submission.shape

(100, 4)